# 傾向スコアを用いた因果推論

Lalondeデータセット（NSW処置群 + CPS比較群）を用いて、傾向スコアによる因果推論を実施する。

## Step 0: 環境構築・データ読み込み

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression

plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

In [2]:
df_org = pd.read_csv('../data/ec675_nsw.tab', sep='\t')
df_org.head()

,treated,age,educ,black,married,nodegree,dwincl,re74,re75,re78,hisp,early_ra,sample
0,NaN,42,16,0,1,0,NaN,0.0000,0.0000,100.485405,0,NaN,2.0
1,NaN,20,13,0,0,0,NaN,2366.7942,3317.4678,4793.745000,0,NaN,2.0
2,NaN,37,12,0,1,0,NaN,25862.3220,22781.8550,25564.670000,0,NaN,2.0
3,NaN,48,12,0,1,0,NaN,21591.1210,20839.3550,20550.744000,0,NaN,2.0
4,NaN,51,12,0,1,0,NaN,21395.1930,21575.1780,22783.588000,0,NaN,2.0


In [3]:
df_org.info()

<class 'pandas.DataFrame'>
RangeIndex: 19204 entries, 0 to 19203
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   treated   722 non-null    float64
 1   age       19204 non-null  int64  
 2   educ      19204 non-null  int64  
 3   black     19204 non-null  int64  
 4   married   19204 non-null  int64  
 5   nodegree  19204 non-null  int64  
 6   dwincl    722 non-null    float64
 7   re74      19204 non-null  float64
 8   re75      19204 non-null  float64
 9   re78      19204 non-null  float64
 10  hisp      19204 non-null  int64  
 11  early_ra  722 non-null    float64
 12  sample    19204 non-null  float64
dtypes: float64(7), int64(6)
memory usage: 1.9 MB


In [4]:
df_org['sample'].value_counts()

sample
2.0    15992
3.0     2490
1.0      722
Name: count, dtype: int64

### 分析用データの作成

- NSW処置群（sample=1, treated=1）とCPS比較群（sample=2）を抽出
- CPS群のtreatedを0に設定

In [ ]:
df_nsw_treated = df_org[(df_org['sample'] == 1) & (df_org['treated'] == 1)].copy()
df_cps = df_org[df_org['sample'] == 2].copy()

df_cps['treated'] = 0

df_analysis = pd.concat([df_nsw_treated, df_cps], ignore_index=True)

print(f"Treated: {df_analysis['treated'].sum()}")
print(f"Control: {(df_analysis['treated'] == 0).sum()}")
print(f"Total: {len(df_analysis)}")

### ベンチマーク効果の算出

NSW RCTデータ（sample=1）の処置群と対照群の差分をATTとして算出

In [ ]:
df_nsw_rct = df_org[df_org['sample'] == 1].copy()

y_treat_rct = df_nsw_rct[df_nsw_rct['treated'] == 1]['re78']
y_control_rct = df_nsw_rct[df_nsw_rct['treated'] == 0]['re78']

att_rct = y_treat_rct.mean() - y_control_rct.mean()

print(f"Benchmark ATT (RCT): ${att_rct:.2f}")

## Step 1: 探索的データ分析（EDA）

### 基本統計量の比較

In [9]:
covariates = ['age', 'educ', 'black', 'hisp', 'married', 'nodegree', 're74', 're75']

df_treat = df_analysis[df_analysis['treated'] == 1]
df_control = df_analysis[df_analysis['treated'] == 0]

stats_treat = df_treat[covariates].describe().T[['mean', 'std']]
stats_control = df_control[covariates].describe().T[['mean', 'std']]

stats_treat.columns = ['mean_treat', 'std_treat']
stats_control.columns = ['mean_control', 'std_control']

df_stats = pd.concat([stats_treat, stats_control], axis=1)
df_stats

,mean_treat,std_treat,mean_control,std_control
age,24.626263,6.686391,33.225238,11.045216
educ,10.380471,1.817712,12.027514,2.870846
black,0.801347,0.399660,0.073537,0.261024
hisp,0.094276,0.292706,0.072036,0.258556
married,0.168350,0.374808,0.711731,0.452971
nodegree,0.730640,0.444376,0.295835,0.456432
re74,3570.998958,5773.133746,14016.800693,9569.796381
re75,3066.098191,4874.888908,13650.803188,9270.402995


### 標準化平均差（SMD）の算出

$$
SMD = \frac{\bar{X}_{treat} - \bar{X}_{control}}{\sqrt{(s^2_{treat} + s^2_{control})/2}}
$$

目安: |SMD| < 0.1 でバランス良好

In [10]:
def calculate_smd(df, covariates, treatment_col='treated'):
    """
    Calculate Standardized Mean Difference for covariates
    """
    df_treat = df[df[treatment_col] == 1]
    df_control = df[df[treatment_col] == 0]
    
    smd_dict = {}
    
    for cov in covariates:
        mean_treat = df_treat[cov].mean()
        mean_control = df_control[cov].mean()
        var_treat = df_treat[cov].var()
        var_control = df_control[cov].var()
        
        pooled_std = np.sqrt((var_treat + var_control) / 2)
        smd = (mean_treat - mean_control) / pooled_std
        
        smd_dict[cov] = smd
    
    return pd.Series(smd_dict)

In [11]:
smd_before = calculate_smd(df_analysis, covariates)

df_smd = pd.DataFrame({
    'SMD_before': smd_before
})

df_smd

,SMD_before
age,-0.941863
educ,-0.685500
black,2.156243
hisp,0.080534
married,-1.307049
nodegree,0.965279
re74,-1.321777
re75,-1.429160


In [ ]:
The SMD values show that many covariates are imbalanced before adjustment (|SMD| > 0.1).

調整前の標準化平均差を確認すると、多くの共変量でバランスが取れていないことがわかる。

## Step 2: 傾向スコアの推定

### ロジスティック回帰による傾向スコアの推定

In [ ]:
print("Summary statistics of propensity scores:")
df_analysis.groupby('treated')['propensity_score'].describe()

In [13]:
print("傾向スコアの基本統計量:")
df_analysis.groupby('treated')['propensity_score'].describe()

傾向スコアの基本統計量:


,count,mean,std,min,25%,50%,75%,max
treated,,,,,,,,
0.0,15992.0,0.013334,0.053969,0.000049,0.000184,0.000807,0.004664,0.566056
1.0,297.0,0.288287,0.183912,0.001088,0.130666,0.296278,0.465921,0.557671


### 傾向スコアの分布の可視化

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

bins = np.arange(0, 1.05, 0.05)

ax.hist(df_analysis[df_analysis['treated'] == 1]['propensity_score'], 
        bins=bins, alpha=0.5, label='Treated', color='blue', edgecolor='black')
ax.hist(df_analysis[df_analysis['treated'] == 0]['propensity_score'], 
        bins=bins, alpha=0.5, label='Control', color='red', edgecolor='black')

ax.set_xlabel('Propensity Score')
ax.set_ylabel('Count')
ax.set_title('Distribution of Propensity Scores')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

bins = np.arange(0, 1.05, 0.05)

df_analysis['ps_bin'] = pd.cut(df_analysis['propensity_score'], bins=bins)

overlap_check = df_analysis.groupby(['ps_bin', 'treated']).size().unstack(fill_value=0)
overlap_check['both_present'] = (overlap_check[0] > 0) & (overlap_check[1] > 0)

print("Common Support Check:")
print(f"Bins with both groups: {overlap_check['both_present'].sum()} / {len(overlap_check)}")
print("\nSample counts by bin:")
overlap_check

In [15]:
bins = np.arange(0, 1.05, 0.05)

df_analysis['ps_bin'] = pd.cut(df_analysis['propensity_score'], bins=bins)

overlap_check = df_analysis.groupby(['ps_bin', 'treated']).size().unstack(fill_value=0)
overlap_check['both_present'] = (overlap_check[0] > 0) & (overlap_check[1] > 0)

print("共通サポート仮定の確認:")
print(f"両群が存在するbin: {overlap_check['both_present'].sum()} / {len(overlap_check)}")
print("\n各binのサンプル数:")
overlap_check

共通サポート仮定の確認:
両群が存在するbin: 12 / 12

各binのサンプル数:


treated,0.0,1.0,both_present
ps_bin,,,
"(0.0, 0.05]",15187,46,True
"(0.05, 0.1]",283,17,True
"(0.1, 0.15]",138,23,True
"(0.15, 0.2]",127,22,True
"(0.2, 0.25]",52,20,True
"(0.25, 0.3]",33,26,True
"(0.3, 0.35]",52,21,True
"(0.35, 0.4]",10,19,True
"(0.4, 0.45]",14,16,True
